# Phase 5: Baseline Models — Telco Customer Churn

## 1. Executive Summary & Objective
In **Phase 5**, we establish benchmark predictive performance across four diverse machine learning algorithms:
1. **Logistic Regression**: Linear baseline with calibrated probabilities.
2. **Decision Tree**: Non-linear, non-parametric tree benchmark.
3. **Random Forest**: Bagged ensemble of decision trees to mitigate variance.
4. **XGBoost**: Scaled gradient boosted decision trees for competitive benchmarking.

### Methodological Safeguards:
- **Stratified Split**: 80% train / 20% test partition maintaining the ~26.5% churn ratio.
- **Zero Data Leakage**: Feature scaling and one-hot encoding fit strictly on training data (`X_train`) and transformed on test data (`X_test`).
- **Comprehensive Metrics**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC.

In [1]:
# 2. Environment Setup & Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn & XGBoost
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# Internal modules
sys.path.append('..')
from src.feature_engineering import engineer_features, get_feature_types, build_preprocessor
from src.train import get_baseline_models, save_model
from src.evaluate import evaluate_model, compare_models

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 3. Data Preparation & Stratified Split

In [2]:
# Load cleaned data and engineer features
data_path = Path('../data/processed/cleaned_churn_data.csv')
df_clean = pd.read_csv(data_path)
df_feat = engineer_features(df_clean)

X = df_feat.drop(columns=['Churn'])
y = df_feat['Churn']

num_cols, cat_cols = get_feature_types(df_feat)

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples (Churn rate: {y_train.mean():.2%})")
print(f"Test set:     {X_test.shape[0]} samples (Churn rate: {y_test.mean():.2%})")
print(f"Feature count: {len(num_cols)} numerical, {len(cat_cols)} categorical.")

Training set: 5634 samples (Churn rate: 26.54%)
Test set:     1409 samples (Churn rate: 26.54%)
Feature count: 9 numerical, 17 categorical.


## 4. Model Training Pipeline
Each classifier is coupled with our `ColumnTransformer` preprocessor into an end-to-end `Pipeline`.

In [3]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

fitted_pipelines = {}
for name, clf in models.items():
    pipe = Pipeline([
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    print(f"[x] Trained: {name}")

[x] Trained: Logistic Regression
[x] Trained: Decision Tree
[x] Trained: Random Forest
[x] Trained: XGBoost


## 5. Baseline Benchmark Results Table
Comparing predictive performance across all 4 baseline algorithms on the unseen test set.

In [4]:
res_df = compare_models(fitted_pipelines, X_test, y_test)
print("=== BASELINE MODEL COMPARISON TABLE ===")
print(res_df.to_string(index=False))

# Export to reports/model_results.csv
results_csv = Path('../reports/model_results.csv')
res_df.to_csv(results_csv, index=False)
print(f"\nSaved results table to {results_csv.resolve()}")

=== BASELINE MODEL COMPARISON TABLE ===
              model  accuracy  precision  recall  f1_score  roc_auc  pr_auc
Logistic Regression    0.7991     0.6542  0.5160    0.5770   0.8426  0.6394
            XGBoost    0.7864     0.6189  0.5080    0.5580   0.8276  0.6114
      Random Forest    0.7991     0.6532  0.5187    0.5782   0.8275  0.6285
      Decision Tree    0.7260     0.4846  0.5053    0.4948   0.6551  0.3762

Saved results table to D:\ML\customer-churn-prediction\reports\model_results.csv


## 6. Confusion Matrix Diagnostics
Analyzing True Positives, False Positives, True Negatives, and False Negatives.

In [5]:
for name in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
    pipe = fitted_pipelines[name]
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print(f"--- {name} Confusion Matrix ---")
    print(f"  True Negatives (Retained correctly): {cm[0, 0]}")
    print(f"  False Positives (Retained mistaken as Churn): {cm[0, 1]}")
    print(f"  False Negatives (Churners missed!): {cm[1, 0]}")
    print(f"  True Positives (Churners caught): {cm[1, 1]}")
    print()

--- Logistic Regression Confusion Matrix ---
  True Negatives (Retained correctly): 933
  False Positives (Retained mistaken as Churn): 102
  False Negatives (Churners missed!): 181
  True Positives (Churners caught): 193

--- Decision Tree Confusion Matrix ---
  True Negatives (Retained correctly): 834
  False Positives (Retained mistaken as Churn): 201
  False Negatives (Churners missed!): 185
  True Positives (Churners caught): 189

--- Random Forest Confusion Matrix ---
  True Negatives (Retained correctly): 932
  False Positives (Retained mistaken as Churn): 103
  False Negatives (Churners missed!): 180
  True Positives (Churners caught): 194

--- XGBoost Confusion Matrix ---
  True Negatives (Retained correctly): 918
  False Positives (Retained mistaken as Churn): 117
  False Negatives (Churners missed!): 184
  True Positives (Churners caught): 190



## 7. Model Persistence
Serializing trained baseline pipelines into `models/` directory.

In [6]:
save_model(fitted_pipelines['Logistic Regression'], '../models/logistic_regression.pkl')
save_model(fitted_pipelines['Random Forest'], '../models/random_forest.pkl')
save_model(fitted_pipelines['Logistic Regression'], '../models/best_model.pkl')
print("[x] Saved models/logistic_regression.pkl, models/random_forest.pkl, models/best_model.pkl")

[x] Saved models/logistic_regression.pkl, models/random_forest.pkl, models/best_model.pkl


## 8. Baseline Analysis & Takeaways for Phase 6

1. **The Recall Bottleneck**: All four baseline models exhibit low recall on churners (~**50.5%–51.9%**). Due to default 0.5 decision thresholds and class imbalance (73.5% vs. 26.5%), models fail to detect nearly half of the customers who churn.
2. **Logistic Regression Leads in Discrimination**: Achieves the highest **ROC-AUC (0.8426)** and highest **PR-AUC (0.6394)**, proving that regularized linear decision boundaries with engineered features are competitive.
3. **Random Forest Leads in F1**: Achieves the highest baseline **F1-Score (0.5782)** with strong precision (65.32%).
4. **Decision Tree Overfitting**: Decision Tree lags behind across every metric (ROC-AUC: **0.6551**), highlighting severe variance when unpruned and un-ensembled.
5. **Requirement for Phase 6**: Applying class imbalance techniques (`class_weight='balanced'`, SMOTE) is essential to elevate recall from ~51% to target levels (>70%).